# Cell Embedding Sanity Check

Zero-training test: does `cell_embedding = normalized_abundance @ gene_embeddings` separate cell types?

**Pipeline:** ESM-2 gene embeddings + Census hematopoietic cells -> protein-informed cell embeddings -> UMAP + silhouette + kNN accuracy

**Requirements:** Runs on cluster. Needs:
- `data/embeddings/gene_to_embedding.pkl`
- Local SOMA database
- Cached ontology (`data/processed/ontology.pkl` -- run `python -m scipher.hierarchy.cache_ontology` first)

In [ ]:
import sys
import logging
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"

from scipher.hierarchy.ontology_utils import load_ontology, get_sub_DAG
from scipher.hierarchy.config import CENSUS_VERSION

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)

sc.settings.set_figure_params(dpi=100, frameon=False)

## 1. Load gene embeddings + build Ensembl->symbol mapping

In [ ]:
# Load ESM-2 gene embeddings (keyed by gene symbol)
EMB_PATH = DATA_DIR / "embeddings" / "gene_to_embedding.pkl"
with open(EMB_PATH, "rb") as f:
    gene_to_embedding = pickle.load(f)

print(f"Gene embeddings: {len(gene_to_embedding):,} genes x {next(iter(gene_to_embedding.values())).shape[0]}-dim")

# Build Ensembl ID -> gene symbol mapping from BioMart
# Census uses Ensembl IDs as var_names, but embeddings are keyed by gene symbol
BIOMART_FILE = DATA_DIR / "raw" / "mart_export.txt"
df_biomart = pd.read_csv(BIOMART_FILE)
df_pc = df_biomart[df_biomart["Gene type"] == "protein_coding"].dropna(subset=["Gene stable ID", "Gene name"])

# Deduplicate: keep first mapping per Ensembl ID
ensembl_to_symbol = dict(zip(df_pc["Gene stable ID"], df_pc["Gene name"]))
print(f"Ensembl->symbol mappings: {len(ensembl_to_symbol):,}")

# How many of those symbols have embeddings?
symbols_with_emb = set(ensembl_to_symbol.values()) & set(gene_to_embedding.keys())
print(f"Symbols with ESM-2 embeddings: {len(symbols_with_emb):,}")

## 2. Load hematopoietic cells from Census

Uses `scipher.hierarchy` ontology utils (copied from McCell) + local SOMA database.

In [ ]:
import tiledbsoma as soma

ROOT_CL_ID = "CL:0000988"  # hematopoietic cell
N_CELLS = 50_000
MIN_CELL_COUNT = 100  # minimum cells per type (lower than McCell's 5000 since we subsample)

# Load ontology from cache
cl = load_ontology()

# Get all descendants of hematopoietic cell
hema_descendants = get_sub_DAG(cl, ROOT_CL_ID)
hema_ids = [t.id for t in hema_descendants]
print(f"Hematopoietic ontology terms: {len(hema_ids)}")

In [ ]:
# Open local SOMA database (same path as McCell's train.ipynb)
SOMA_URI = "/scratch/sigbio_project_root/sigbio_project25/jingqiao/mccell-single/soma_db_homo_sapiens"
print(f"Opening SOMA database: {SOMA_URI}")

experiment = soma.open(SOMA_URI, mode="r")

# Step 1: Read obs metadata to find hematopoietic cells
# Same filter pattern as McCell's data_loader.py
obs_df = (
    experiment.obs.read(
        value_filter='assay == "10x 3\' v3" and is_primary_data == True',
        column_names=["soma_joinid", "cell_type_ontology_term_id", "cell_type"],
    )
    .concat()
    .to_pandas()
)
print(f"Total 10x v3 primary cells: {len(obs_df):,}")

# Filter to hematopoietic descendants
obs_df = obs_df[obs_df["cell_type_ontology_term_id"].isin(hema_ids)].copy()
print(f"Hematopoietic cells: {len(obs_df):,}")

# Drop rare cell types
type_counts = obs_df["cell_type"].value_counts()
keep_types = type_counts[type_counts >= MIN_CELL_COUNT].index
obs_df = obs_df[obs_df["cell_type"].isin(keep_types)].copy()
print(f"After dropping types with < {MIN_CELL_COUNT} cells: {len(obs_df):,} cells, {obs_df['cell_type'].nunique()} types")

# Subsample
if len(obs_df) > N_CELLS:
    obs_df = obs_df.sample(n=N_CELLS, random_state=42)
    print(f"Subsampled to {len(obs_df):,} cells")

print(f"\nCell type distribution:")
print(obs_df["cell_type"].value_counts().head(20).to_string())

In [ ]:
# Step 2: Read expression data for subsampled cells
# Use protein-coding gene filter from BioMart (same pattern as McCell's train.ipynb)
gene_list = df_pc["Gene stable ID"].unique().tolist()
var_value_filter = f"feature_id in {gene_list}"

joinids = obs_df["soma_joinid"].tolist()

with experiment.axis_query(
    measurement_name="RNA",
    obs_query=soma.AxisQuery(coords=(joinids,)),
    var_query=soma.AxisQuery(value_filter=var_value_filter),
) as query:
    adata = query.to_anndata(
        X_name="raw",
        column_names={"obs": ["cell_type_ontology_term_id", "cell_type"]},
    )

print(f"Loaded AnnData: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes")
print(f"Cell types: {adata.obs['cell_type'].nunique()}")

## 3. Remap var_names + create protein-informed cell embeddings

In [ ]:
# Remap AnnData var_names from Ensembl IDs to gene symbols
# Census var_names are Ensembl IDs (e.g., ENSG00000012048)
# Gene embeddings are keyed by gene symbol (e.g., BRCA1)

original_var_names = adata.var_names.tolist()
new_var_names = [ensembl_to_symbol.get(eid, eid) for eid in original_var_names]

# Handle duplicates: if multiple Ensembl IDs map to the same symbol, keep the first
seen = set()
dup_mask = []
for name in new_var_names:
    if name in seen:
        dup_mask.append(False)
    else:
        seen.add(name)
        dup_mask.append(True)

n_dups = sum(1 for x in dup_mask if not x)
if n_dups > 0:
    adata = adata[:, dup_mask].copy()
    new_var_names = [n for n, keep in zip(new_var_names, dup_mask) if keep]

adata.var_names = new_var_names
adata.var_names_make_unique()

# Stats
n_mapped = sum(1 for name in adata.var_names if name in gene_to_embedding)
n_unmapped_symbol = sum(1 for name in adata.var_names if name.startswith("ENSG"))
n_no_embedding = sum(1 for name in adata.var_names if not name.startswith("ENSG") and name not in gene_to_embedding)

print(f"Gene remapping results:")
print(f"  Total genes in AnnData:      {adata.n_vars:,}")
print(f"  Mapped to symbol + embedding: {n_mapped:,} ({100*n_mapped/adata.n_vars:.1f}%)")
print(f"  No symbol (still Ensembl ID): {n_unmapped_symbol:,}")
print(f"  Symbol but no embedding:      {n_no_embedding:,}")
print(f"  Duplicates removed:           {n_dups:,}")

In [ ]:
# Create protein-informed cell embeddings via WeightedSumEmbedder
from scipher.embedders.weighted_sum import WeightedSumEmbedder

embedder = WeightedSumEmbedder(gene_to_embedding)
cell_embeddings = embedder.embed(adata)

print(f"Cell embeddings: {cell_embeddings.shape}")
print(f"  Mean norm: {np.linalg.norm(cell_embeddings, axis=1).mean():.2f}")
print(f"  Any NaN:   {np.isnan(cell_embeddings).any()}")

# Store in adata for downstream use
adata.obsm["X_protein"] = cell_embeddings

## 4. Baseline: standard scanpy expression pipeline

In [ ]:
# Standard expression-only pipeline for baseline comparison
# Work on a copy so we don't overwrite raw counts needed by WeightedSumEmbedder
adata_expr = adata.copy()

sc.pp.normalize_total(adata_expr, target_sum=1e4)
sc.pp.log1p(adata_expr)
sc.pp.highly_variable_genes(adata_expr, n_top_genes=2000)
sc.pp.pca(adata_expr, n_comps=50)
sc.pp.neighbors(adata_expr, n_pcs=50)
sc.tl.umap(adata_expr)

# Save expression UMAP coords back to main adata
adata.obsm["X_umap_expr"] = adata_expr.obsm["X_umap"].copy()
adata.obsm["X_pca"] = adata_expr.obsm["X_pca"].copy()

print("Expression baseline pipeline done")
print(f"  PCA: {adata_expr.obsm['X_pca'].shape}")
print(f"  UMAP: {adata_expr.obsm['X_umap'].shape}")

## 5. UMAP on protein-informed embeddings

In [ ]:
# Compute UMAP from protein-informed cell embeddings
sc.pp.neighbors(adata, use_rep="X_protein")
sc.tl.umap(adata)
adata.obsm["X_umap_protein"] = adata.obsm["X_umap"].copy()

print("Protein-informed UMAP done")

In [ ]:
# Side-by-side UMAPs
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Expression baseline
adata.obsm["X_umap"] = adata.obsm["X_umap_expr"]
sc.pl.umap(adata, color="cell_type", ax=axes[0], show=False, title="Expression (PCA -> UMAP)",
           legend_loc="on data", legend_fontsize=6, frameon=False)

# Protein-informed
adata.obsm["X_umap"] = adata.obsm["X_umap_protein"]
sc.pl.umap(adata, color="cell_type", ax=axes[1], show=False, title="Protein-informed (ESM-2 -> UMAP)",
           legend_loc="on data", legend_fontsize=6, frameon=False)

plt.tight_layout()
plt.savefig(DATA_DIR / "reports" / "cell_embedding_umap_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to data/reports/cell_embedding_umap_comparison.png")

## 6. Quantitative comparison

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder

labels = adata.obs["cell_type"].values
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

# --- Silhouette scores ---
# Subsample for silhouette (expensive for large N)
N_SIL = min(10_000, adata.shape[0])
rng = np.random.RandomState(42)
sil_idx = rng.choice(adata.shape[0], N_SIL, replace=False)

sil_protein = silhouette_score(adata.obsm["X_protein"][sil_idx], labels_encoded[sil_idx])
sil_expr = silhouette_score(adata.obsm["X_pca"][sil_idx], labels_encoded[sil_idx])

print("=== Silhouette Scores ===")
print(f"  Expression (PCA 50-dim):     {sil_expr:.4f}")
print(f"  Protein-informed (1280-dim): {sil_protein:.4f}")

# --- kNN accuracy ---
X_protein = adata.obsm["X_protein"]
X_expr = adata.obsm["X_pca"]

X_train_p, X_test_p, y_train, y_test = train_test_split(
    X_protein, labels_encoded, test_size=0.2, random_state=42, stratify=labels_encoded
)
X_train_e, X_test_e, _, _ = train_test_split(
    X_expr, labels_encoded, test_size=0.2, random_state=42, stratify=labels_encoded
)

knn_protein = KNeighborsClassifier(n_neighbors=15)
knn_protein.fit(X_train_p, y_train)
acc_protein = knn_protein.score(X_test_p, y_test)

knn_expr = KNeighborsClassifier(n_neighbors=15)
knn_expr.fit(X_train_e, y_train)
acc_expr = knn_expr.score(X_test_e, y_test)

print(f"\n=== kNN Accuracy (k=15, 80/20 split) ===")
print(f"  Expression (PCA 50-dim):     {acc_expr:.4f}")
print(f"  Protein-informed (1280-dim): {acc_protein:.4f}")

In [ ]:
# Per-cell-type breakdown
pred_protein = knn_protein.predict(X_test_p)
pred_expr = knn_expr.predict(X_test_e)

type_names = le.classes_
rows = []
for i, ct in enumerate(type_names):
    mask = y_test == i
    n = mask.sum()
    if n == 0:
        continue
    acc_e = (pred_expr[mask] == y_test[mask]).mean()
    acc_p = (pred_protein[mask] == y_test[mask]).mean()
    rows.append({"cell_type": ct, "n_test": n, "knn_expr": acc_e, "knn_protein": acc_p, "diff": acc_p - acc_e})

df_results = pd.DataFrame(rows).sort_values("diff", ascending=False)
print("Per-cell-type kNN accuracy:")
print(df_results.to_string(index=False, float_format="{:.3f}".format))

# Summary
n_protein_wins = (df_results["diff"] > 0).sum()
n_expr_wins = (df_results["diff"] < 0).sum()
n_ties = (df_results["diff"] == 0).sum()
print(f"\nProtein wins: {n_protein_wins}, Expression wins: {n_expr_wins}, Ties: {n_ties}")